As a resort company, I want to evaluate shark attack risk near beach destinations so I can decide where to invest, what activities to promote, and how to communicate safety to guests.

Questions:
1- Which resort regions have lower incident rates?
2- Are attacks concentrated during certain months?
3- Should some water activities be promoted more than others?

In [1]:
!pip install xlrd

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [2]:
"""Load the GSAF shark attack dataset from the Excel file into a pandas DataFrame."""
import pandas as pd
import numpy as np
df = pd.read_excel("GSAF5.xls")
display(df)

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,...,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
0,18th March,2026.0,Unprovoked,USA,California,Big River Beach Mendocino County,Surfing,Unknown,M,39,...,Great White Shark,US Sun: Mendocino Coast News:Melissa Michaelson:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,14th March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Swimming,Unknown,F,?,...,Unknown shark 5ft (1.5m),People Magazine: Kevin McMurray Trackingsharks...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10th March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Wing Foiling,Dave Daniell,M,?,...,Great White Shark 10ft (3m),Perth Now: Kevin McMurray Trackingsharks.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5th March,2026.0,Unprovoked,Australia,Queensland,Lady Elliott Island,Snorkeling,Unknown,M,50's,...,Unknown,News.com.au: ABC News: Andy Currie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,22nd February,2026.0,Unprovoked,New Caledonia,Noumea,Anse Vata near Point Magnin,Wing Foiling,Cyril Chevalier,M,55,...,Tiger or bull shark implicated,Johannes Marchand: Andy Currie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7077,Before 1903,0.0,Unprovoked,AUSTRALIA,Western Australia,Roebuck Bay,Diving,male,M,NaN,...,NaN,"H. Taunton; N. Bartlett, p. 234",ND-0005-RoebuckBay.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0005,ND.0005,6.0,NaN,NaN
7078,Before 1903,0.0,Unprovoked,AUSTRALIA,Western Australia,NaN,Pearl diving,Ahmun,M,NaN,...,NaN,"H. Taunton; N. Bartlett, pp. 233-234",ND-0004-Ahmun.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0004,ND.0004,5.0,NaN,NaN
7079,1900-1905,0.0,Unprovoked,USA,North Carolina,Ocracoke Inlet,Swimming,Coast Guard personnel,M,NaN,...,NaN,"F. Schwartz, p.23; C. Creswell, GSAF",ND-0003-Ocracoke_1900-1905.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0003,ND.0003,4.0,NaN,NaN
7080,1883-1889,0.0,Unprovoked,PANAMA,NaN,"Panama Bay 8ºN, 79ºW",NaN,Jules Patterson,M,NaN,...,NaN,"The Sun, 10/20/1938",ND-0002-JulesPatterson.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0002,ND.0002,3.0,NaN,NaN


In [3]:
"""Inspect the dataset to understand its overall structure:
- the column names
- the data types of each column
- the number of missing values
-the number of duplicated rows
- possible formatting problems and irrelevant columns
"""
print(df.shape)
print("\n")
print(df.columns)
print("\n")
print(df.dtypes)
print("\n")
print(df.isnull().sum())
print("\n")
df.duplicated().sum()

(7082, 23)


Index(['Date', 'Year', 'Type', 'Country', 'State', 'Location', 'Activity',
       'Name', 'Sex', 'Age', 'Injury', 'Fatal Y/N', 'Time', 'Species ',
       'Source', 'pdf', 'href formula', 'href', 'Case Number', 'Case Number.1',
       'original order', 'Unnamed: 21', 'Unnamed: 22'],
      dtype='object')


Date               object
Year              float64
Type               object
Country            object
State              object
Location           object
Activity           object
Name               object
Sex                object
Age                object
Injury             object
Fatal Y/N          object
Time               object
Species            object
Source             object
pdf                object
href formula       object
href               object
Case Number        object
Case Number.1      object
original order    float64
Unnamed: 21        object
Unnamed: 22        object
dtype: object


Date                 0
Year                 2
Type                

np.int64(0)

In [4]:
"""
Standardize column names for easier analysis:
- remove extra spaces
- convert names to lowercase
- replace spaces with underscores
"""

df.columns=df.columns.str.strip().str.lower().str.replace(" ","_")
df = df.rename(columns={"fatal_y/n": "fatal_y_n"})

In [5]:
"""
Remove columns that are empty or not useful for the analysis.
"""

df=df.drop(columns=["unnamed:_21", "unnamed:_22", "href","href_formula","pdf","original_order"],errors="ignore")


In [6]:
"""check columns's values and clean and standardize the columns.

The values in columns may contain:
- extra spaces
- lowercase and uppercase variations
- invalid symbols
- missing values """

df.sex.unique()

array(['M', 'F', 'M ', '?', 'F ', nan, ' M', 'm', 'lli', 'M x 2', 'N',
       '.'], dtype=object)

In [7]:
df["sex"] = (df["sex"].astype(str).str.strip().str.upper().replace({"NAN":np.nan,"?": "Unknown",".": "Unknown", "N": "Unknown", "LLI": "Unknown","M X 2": "Unknown"}))
df.sex.unique()

array(['M', 'F', 'Unknown', nan], dtype=object)

In [8]:
"""
This column should indicate whether an attack was fatal ('Y') or non-fatal ('N'),
"""
df.fatal_y_n.unique()

array(['N', 'Y', 'F', 'M', nan, 'n', 'Nq', 'UNKNOWN', 2017, 'Y x 2', ' N',
       'N ', 'y'], dtype=object)

In [9]:
df["fatal_y_n"] = (df["fatal_y_n"].astype(str).str.strip().str.upper().replace({"F": "Unknown","M": "Unknown","2017": "Unknown", "Y X 2": "Unknown", "NQ":"Unknown","NAN":np.nan,"UNKNOWN":"Unknown"}))
df.fatal_y_n.unique()

array(['N', 'Y', 'Unknown', nan], dtype=object)

In [10]:
for col in ["country", "state", "location"]:
 df[col] = df[col].astype(str).str.strip().replace("nan", np.nan)
 df[col] = df[col].str.title()

"""
Group detailed activities into broader activity categories.

The raw activity descriptions are too detailed and inconsistent for direct
comparison, so they are grouped into broader categories such as:
- Surfing
- Swimming/Wading
- Diving/Snorkeling
- Fishing
- Boating/Watercraft
- Other
"""

df["activity"] = df["activity"].astype(str).str.strip().replace("nan", np.nan) 

def group_activity(activity): 
    if pd.isna(activity): 
        return np.nan

    activity = activity.lower()

    if "surf" in activity:
        return "Surfing"
    elif "swim" in activity or "bathing" in activity or "wading" in activity:
        return "Swimming/Wading"
    elif "dive" in activity or "snork" in activity:
        return "Diving/Snorkeling"
    elif "fish" in activity:
        return "Fishing"
    elif "boat" in activity or "kayak" in activity or "canoe" in activity or "sail" in activity:
        return "Boating/Watercraft"
    else:
        return "Other"
df["activity_group"] = df["activity"].apply(group_activity) 
print(df["activity_group"].value_counts()) 


activity_group
Other                 1736
Swimming/Wading       1652
Surfing               1462
Fishing               1296
Boating/Watercraft     185
Diving/Snorkeling      168
Name: count, dtype: int64


In [11]:
df.year.unique()
df.time.unique()

array(['1715hrs', '1015hrs', '0800hrs', '0815hrs', '?', '1100hrs',
       '1815hrs', '0830hrs', '1145hrs', '1820hrs', '1620hrs', '0540hrs',
       '1628hrs', '1200hrs', '1630hrs', '0630hrs', '1745hrs',
       'Not stated', 'Mid afternoon', '1800hrs', '1823 hrs', '1330hrs',
       '0100hrs', '0930hrs', '1524hrs', '0730hrs', '1300hrs', '1055 hrs',
       '1130hrs', 'PM', '1150hrs', '1500hrs', '1600', '1615 hrs',
       '1210hrs', '1211 hrs', '1400hrs', 'pm', '0945hrs', '1430hrs',
       '1210 hrs', '1340hrs', 'Unknown', '1503hrs', '1830hrs', '1645 hrs',
       '1711hrs', '1600hrs', '1615hr', '1710hr', '1637hr', 'AM', '1600hr',
       '1735hrs', '1115hrs', '1118hrs', '1615hrs', '1100hr',
       'after 1200hr', '1400hr', 15.5, '13h15', '9h', 1300, '14h',
       '15h30', '13h30', '9h15', 'Not advised', '13h40', '12h30', '16h00',
       nan, '11h30', '06h30', '20h00', '13h00', '11h12', '16h30', '15h00',
       '02h00', '09h15', 'Early Morning', '16h32', '11h00', 'Morning',
       '10h30', '1

In [12]:
df["time"] = df["time"].replace(r"^\s*$", np.nan, regex=True)
df["time"] = df["time"].astype(str).str.strip().str.lower()
df["time"] = df["time"].replace({"nan": np.nan,"?": np.nan,"--": np.nan,"unknown": np.nan,"not stated": np.nan,"not advised": np.nan,"x": np.nan, "n": np.nan,"pm": "afternoon","p.m.": "afternoon", "am": "morning", "a.m.": "morning","midday": "noon",'"midday"': "noon","midday.": "noon","noon": "noon","morning ": "morning",
    "early  morning": "early morning","mid morning": "mid morning","mid-morning": "mid morning", "early afternoon": "afternoon","late afternoon": "afternoon","late afternon": "afternoon","late afternoon": "afternoon","evening": "evening",'"evening"': "evening",'"night"': "night","night": "night","late night": "night","after dark": "night",'"after dark"': "night",
    "dark": "night", "after dusk": "evening","dusk": "evening","sunset": "evening","nightfall": "evening","dawn": "early morning","daybreak": "early morning", "before daybreak": "early morning","just before dawn": "early morning", "daytime": "afternoon","lunchtime": "noon"

})



In [13]:
df.species.unique()

array(['Great White Shark', 'Unknown shark 5ft (1.5m)',
       'Great White Shark 10ft (3m)', ..., "12' tiger shark",
       'Blue pointers',
       'Said to involve a grey nurse shark that leapt out of the water and  seized the boy but species identification is questionable'],
      dtype=object)

In [14]:
"""
Clean the raw 'species' column.

The shark species descriptions are inconsistent and may include:
- species names
- size estimates
- uncertainty
- non-confirmed shark involvement

This step standardizes the text and prepares it for grouping into
broader shark categories.
"""

df["species"] = df["species"].astype(str).str.strip().replace("nan", np.nan)
def group_species(species):
    if pd.isna(species):
        return np.nan

    s = species.lower()

    if "white" in s:
        return "Great White"
    elif "tiger" in s:
        return "Tiger"
    elif "bull" in s:
        return "Bull"
    elif "unknown" in s:
        return "Unknown Shark"
    elif "not confirmed" in s:
        return "Not Confirmed"
    elif "shark" in s:
        return "Other Shark"
    else:
        return "Other"

df["species_group"] = df["species"].apply(group_species)
print(df["species_group"].value_counts())

species_group
Other Shark      2066
Great White       759
Tiger             347
Other             292
Bull              237
Not Confirmed     229
Unknown Shark      21
Name: count, dtype: int64


In [15]:

"""
Prepare the columns directly relevant to :
- identifying safer resort regions
- analyzing attacks by month
- comparing risk across water activities

Cleaning decisions:
- Rows with missing country are dropped because country is essential for regional analysis and missing values are a few.
- Missing state and location values are filled with 'Unknown' (we can not guess the state and location)so those rows can still be used.
- The date column is converted to datetime so month-based analysis becomes possible.
- The activity column is cleaned, but missing values are not guessed or imputed.Missing values in the activity column were not filled using the mode, because this would artificially assign a behavior to incidents where the activity is unknown. 
Since activity is one of the main analytical variables in this project, imputing it with the most frequent category would bias the results and potentially lead to misleading business conclusions. Instead, missing values were either kept as missing or labeled as "Unknown".
"""
df = df.dropna(subset=["country"])
df["state"] = df["state"].fillna("Unknown")
df["location"] = df["location"].fillna("Unknown")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

/var/folders/79/dtmd18xj3dz5rnfz6nz39djw0000gn/T/ipykernel_36320/3084204811.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["date"], errors="coerce")


1- Which resort regions have lower incident rates?

The dataset is first explored using df.head(), df.columns, and df.info(). 

These functions help identify the structure of the data and determine which column represents location information, such as Country, Area, or Location.

In [16]:
# count shark attacks per country
region_counts = df.groupby('country').size().sort_values()

In [17]:
# Filters the data to keep only regions with more than 10 shark attacks
region_counts = region_counts[region_counts > 10]

In [18]:
# Since the data is sorted in ascending order, this shows the regions with the lowest number of shark attacks
region_counts.head(10)

country
Columbia                11
Samoa                   11
Ecuador                 11
Senegal                 11
Venezuela               11
United Kingdom          11
Turkey                  12
South Atlantic Ocean    12
Iraq                    12
Thailand                13
dtype: int64

These regions represent comparatively lower shark attack risk, which could inform both operational planning and guest communications. While safety is just one factor in destination choice, highlighting low-risk areas may enhance guest confidence and support marketing strategies.

2- Are attacks concentrated during certain months?

In [19]:
df["month"] = df["date"].dt.month_name()
month_counts = df["month"].value_counts()
print(month_counts)

month
January      799
July         692
August       599
September    541
June         494
October      456
April        450
December     446
March        422
May          418
November     416
February     388
Name: count, dtype: int64


The most concentrated attack happened in January and July

3- Should some water activities be promoted more than others?

In [20]:
activity_counts = df["activity_group"].value_counts(dropna=True,normalize=True)*100
print(activity_counts)

activity_group
Other                 26.502478
Swimming/Wading       25.433705
Surfing               22.630112
Fishing               20.043371
Boating/Watercraft     2.803594
Diving/Snorkeling      2.586741
Name: proportion, dtype: float64


The highest share of incidents falls under Other (26.5%), followed closely by Swimming/Wading (25.4%), Surfing (22.6%), and Fishing (20.0%). In contrast, Boating/Watercraft (2.8%) and Diving/Snorkeling (2.6%) account for a much smaller proportion of recorded incidents.

From a resort business perspective, this suggests that:
	•	Swimming/Wading and Surfing may require stronger safety communication, supervision, warning signage, or more cautious promotion.
	•	Fishing should also be treated carefully, as it represents a relatively large share of incidents.
	•	Boating/Watercraft and Diving/Snorkeling appear less frequently in the incident data and may be safer options to promote, although          they should still include clear safety guidance.